In [1]:
import pandas as pd
import altair as alt

url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv"

df = pd.read_csv(url)

print(df.shape)
print(df.columns.tolist())
df.head()

(4747, 29)
['observed', 'location_details', 'county', 'state', 'season', 'title', 'latitude', 'longitude', 'date', 'number', 'classification', 'geohash', 'temperature_high', 'temperature_mid', 'temperature_low', 'dew_point', 'humidity', 'cloud_cover', 'moon_phase', 'precip_intensity', 'precip_probability', 'precip_type', 'pressure', 'summary', 'uv_index', 'visibility', 'wind_bearing', 'wind_speed', 'location']


,observed,location_details,county,state,season,title,latitude,longitude,date,number,...,precip_intensity,precip_probability,precip_type,pressure,summary,uv_index,visibility,wind_bearing,wind_speed,location
0,Ed L. was salmon fishing with a companion in P...,East side of Prince William Sound,Valdez-Chitina-Whittier County,Alaska,Fall,NaN,NaN,NaN,NaN,1261.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,heh i kinda feel a little dumb that im reporti...,"the road is off us rt 80, i dont know the exit...",Warren County,New Jersey,Fall,NaN,NaN,NaN,NaN,438.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I was on my way to Claremont from Lebanon on R...,Close to Claremont down 120 not far from Kings...,Sullivan County,New Hampshire,Summer,Report 55269: Dawn sighting at Stevens Brook o...,43.41549,-72.33093,2016-06-07,55269.0,...,0.001,0.7,rain,998.87,Mostly cloudy throughout the day.,6.0,9.70,262.0,0.49,POINT(-72.33093000000001 43.415490000000005)
3,I was northeast of Macy Nebraska along the Mis...,Latitude & Longitude : 42.158230 -96.344197,Thurston County,Nebraska,Spring,Report 59757: Possible daylight sighting of a ...,42.15685,-96.34203,2018-05-25,59757.0,...,0.000,0.0,NaN,1008.07,Partly cloudy in the morning.,10.0,8.25,193.0,3.33,POINT(-96.34203000000001 42.15685)
4,"While this incident occurred a long time ago, ...","Ward County, Just outside of a the Minuteman T...",Ward County,North Dakota,Spring,Report 751: Hunter describes described being s...,48.25422,-101.31660,2000-04-21,751.0,...,NaN,NaN,rain,1011.47,Partly cloudy until evening.,6.0,10.00,237.0,11.14,POINT(-101.3166 48.254220000000004)


In [2]:
# Count sightings by state
state_counts = (
    df.dropna(subset=['state'])
    .groupby('state')
    .size()
    .reset_index(name='Sightings')
    .sort_values('Sightings', ascending=False)
    .head(15)
)

# Create bar chart
chart1 = alt.Chart(state_counts).mark_bar(
    color='#4682B4'
).encode(
    x=alt.X(
        'Sightings:Q',
        title='Number of Sightings'
    ),
    y=alt.Y(
        'state:N',
        sort='-x',
        title='State'
    ),
    tooltip=[
        alt.Tooltip('state:N', title='State'),
        alt.Tooltip('Sightings:Q', title='Sightings')
    ]
).properties(
    title='Top 15 States by Reported Bigfoot Sightings',
    width=650,
    height=450
)

chart1

alt.Chart(...)

In [3]:
# Convert dates to datetime
df['date_clean'] = pd.to_datetime(df['date'], errors='coerce')

# Remove rows missing dates or states
df_time = df.dropna(subset=['date_clean', 'state']).copy()

# Extract year
df_time['Year'] = df_time['date_clean'].dt.year

# Count sightings by year and state
yearly_counts = (
    df_time.groupby(['Year', 'state'])
    .size()
    .reset_index(name='Sightings')
)

# Create dropdown containing all states
states = sorted(yearly_counts['state'].unique().tolist())

dropdown = alt.binding_select(
    options=states,
    name='Select State: '
)

state_selection = alt.selection_point(
    fields=['state'],
    bind=dropdown,
    value=states[0]
)

# Create interactive line chart
chart2 = (
    alt.Chart(yearly_counts)
    .mark_line(point=True, color='#E07A5F')
    .encode(
        x=alt.X('Year:O', title='Year'),
        y=alt.Y('Sightings:Q', title='Number of Sightings'),
        tooltip=[
            alt.Tooltip('state:N', title='State'),
            alt.Tooltip('Year:O', title='Year'),
            alt.Tooltip('Sightings:Q', title='Sightings')
        ]
    )
    .transform_filter(state_selection)
    .add_params(state_selection)
    .properties(
        title='Reported Bigfoot Sightings Over Time by State',
        width=700,
        height=400
    )
)

chart2

alt.Chart(...)

In [4]:
from pathlib import Path

# Create a folder for visualization files
output_folder = Path("assets/json")
output_folder.mkdir(parents=True, exist_ok=True)

# Export both Altair charts
chart1.save(output_folder / "homework7_chart1.json")
chart2.save(output_folder / "homework7_chart2.json")

print("Both charts exported successfully!")

Both charts exported successfully!
